### Data Pre Processing

*   Extract pose, left hand and right hand keypoints using MediaPipe
*   Flip videos horizontally to double the dataset size
*   Flip videos horizontally to double the dataset size

#### Use Python 3.11.9

In [41]:
!pip install "tensorflow>=2.16,<3" tensorflow-metal \
            "mediapipe>=0.10,<0.11" "opencv-python>=4.10,<5" \
            "numpy>=2.0,<2.3" "scikit-learn>=1.4,<1.6" "scipy>=1.11" matplotlib --quiet



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [42]:
!pip install --upgrade mediapipe opencv-python numpy matplotlib --quiet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## 1. Import and Install Dependencies

In [43]:
import cv2, numpy as np
import os
import shutil
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

## 2. Keypoints using MP Holistic

In [48]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [49]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB - for mediapipe
    image.flags.writeable = False                  # Image is no longer writeable - saves a bit of memory
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR - for opencv
    return image, results

In [50]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

In [51]:
def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

In [52]:
# Actions that we try to detect
actions = np.array(['MBS', 'Bedok', 'Clementi', 'Orchard', 'Esplanade', 'City Hall'])
# actions = np.array(['1', '2'])

# Thirty videos worth of data
no_sequences = 15

# Videos are going to be 30 frames in length
sequence_length = 60

# Folder start
start_folder = 0  # or compute from existing dirs if resuming

# Path where video clips are stored
VIDEO_DATA_PATH = os.path.join('MP_Videos_All')

## 3. Data Augmentation

In [53]:

# Use VIDEO_DATA_PATH defined earlier; fall back to 'MP_Videos' if not found
source_dir = VIDEO_DATA_PATH if 'VIDEO_DATA_PATH' in globals() else 'MP_Videos_All'
if not os.path.isdir(source_dir):
    alt = 'MP_Videos_All'
    if os.path.isdir(alt):
        source_dir = alt
        print(f"[INFO] VIDEO_DATA_PATH not found. Using '{source_dir}'.")
    else:
        raise FileNotFoundError(f"Source directory not found: {source_dir}")

# Output directory: mirror of source with '_aug' suffix
output_dir = f"{os.path.normpath(source_dir)}_aug"
os.makedirs(output_dir, exist_ok=True)

supported_exts = {'.mp4', '.avi', '.mov', '.mkv'}
num_original_copied = 0
num_flipped_written = 0
num_errors = 0

print(f"[START] Augmenting videos by horizontal flip\n- Source: {source_dir}\n- Output: {output_dir}")


def open_writer_for_path(path: str, width: int, height: int, fps: float):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.mp4':
        writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        if writer.isOpened():
            return writer, path
        # Fallback to AVI if MP4 fails
        alt_path = os.path.splitext(path)[0] + '.avi'
        writer = cv2.VideoWriter(alt_path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))
        return writer, alt_path
    else:
        writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))
        return writer, path


for root, dirs, files in os.walk(source_dir):
    rel_dir = os.path.relpath(root, source_dir)
    dest_dir = os.path.join(output_dir, rel_dir) if rel_dir != '.' else output_dir
    os.makedirs(dest_dir, exist_ok=True)

    # Filter supported video files in this source directory
    video_files = [f for f in files if os.path.splitext(f)[1].lower() in supported_exts]
    if not video_files:
        continue

    # 1) Copy originals to the mirrored output directory
    for fname in video_files:
        src_path = os.path.join(root, fname)
        dest_orig = os.path.join(dest_dir, fname)
        if not os.path.exists(dest_orig):
            try:
                shutil.copy2(src_path, dest_orig)
                num_original_copied += 1
            except Exception as e:
                print(f"[ERROR] Copy failed for {src_path}: {e}")
                num_errors += 1

    # 2) Determine starting index for flipped videos in this folder
    existing_numbers = set()
    for f in os.listdir(dest_dir):
        base, ext = os.path.splitext(f)
        if ext.lower() in supported_exts and base.isdigit():
            existing_numbers.add(int(base))

    if 'no_sequences' in globals() and isinstance(no_sequences, int):
        start_idx = int(no_sequences)  # e.g., 15
    else:
        start_idx = (max(existing_numbers) + 1) if existing_numbers else 0

    # 3) Write flipped videos with numeric names starting from start_idx
    def numeric_key(name: str):
        base, _ = os.path.splitext(name)
        return int(base) if base.isdigit() else float('inf')

    for fname in sorted(video_files, key=numeric_key):
        src_path = os.path.join(root, fname)

        cap = cv2.VideoCapture(src_path)
        if not cap.isOpened():
            print(f"[WARN] Could not open video: {src_path}")
            num_errors += 1
            cap.release()
            continue

        width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
        fps_in = cap.get(cv2.CAP_PROP_FPS)
        fps_out = fps_in if fps_in and fps_in > 0 else 20.0

        # Find next available numeric filename in destination
        while True:
            out_name = f"{start_idx}{os.path.splitext(fname)[1].lower()}"
            flip_path = os.path.join(dest_dir, out_name)
            if not os.path.exists(flip_path):
                break
            start_idx += 1

        writer, actual_path = open_writer_for_path(flip_path, width, height, fps_out)
        if not writer or not writer.isOpened():
            print(f"[WARN] Could not open writer for {flip_path}")
            cap.release()
            continue

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            flipped = cv2.flip(frame, 1)  # horizontal flip
            writer.write(flipped)

        writer.release()
        cap.release()
        num_flipped_written += 1
        start_idx += 1

        if actual_path != flip_path:
            print(f"[INFO] Fallback writer used. Flipped saved as: {actual_path}")

print(f"[DONE] Originals copied: {num_original_copied}, Flipped written: {num_flipped_written}, Errors: {num_errors}")
print(f"[OUT] Augmented dataset available at: {output_dir}")



[START] Augmenting videos by horizontal flip
- Source: MP_Videos_All
- Output: MP_Videos_All_aug


KeyboardInterrupt: 

## 4. Annotate Videos

In [55]:
import os
import cv2
import numpy as np

# Source (augmented) videos
source_dir = 'MP_Videos_All'
if not os.path.isdir(source_dir):
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

# Output: annotated videos (mirrored tree)
annotated_dir = f"{source_dir}_annotated"
os.makedirs(annotated_dir, exist_ok=True)

supported_exts = {'.mp4', '.avi', '.mov', '.mkv'}


def open_writer_with_fallback(path: str, width: int, height: int, fps: float):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.mp4':
        writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
        if writer.isOpened():
            return writer, path
        alt_path = os.path.splitext(path)[0] + '.avi'
        writer = cv2.VideoWriter(alt_path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))
        return writer, alt_path
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (width, height))
    return writer, path


num_processed = 0
num_annotated_saved = 0
num_errors = 0

print(f"[START] Annotating videos\n- Source: {source_dir}\n- Annotated out: {annotated_dir}")

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for root, dirs, files in os.walk(source_dir):
        rel_dir = os.path.relpath(root, source_dir)
        ann_dir = os.path.join(annotated_dir, rel_dir) if rel_dir != '.' else annotated_dir
        os.makedirs(ann_dir, exist_ok=True)

        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in supported_exts:
                continue

            src_path = os.path.join(root, fname)
            ann_path = os.path.join(ann_dir, fname)

            cap = cv2.VideoCapture(src_path)
            if not cap.isOpened():
                print(f"[WARN] Could not open video: {src_path}")
                num_errors += 1
                continue

            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
            fps_in = cap.get(cv2.CAP_PROP_FPS)
            fps_out = fps_in if fps_in and fps_in > 0 else 20.0

            writer, actual_path = open_writer_with_fallback(ann_path, width, height, fps_out)
            if not writer or not writer.isOpened():
                print(f"[WARN] Could not open writer for {ann_path}")
                cap.release()
                num_errors += 1
                continue

            while True:
                ok, frame = cap.read()
                if not ok:
                    break

                image, results = mediapipe_detection(frame, holistic)
                draw_styled_landmarks(image, results)

                writer.write(image)

            writer.release()
            cap.release()

            num_processed += 1
            num_annotated_saved += 1

print(f"[DONE] Videos processed: {num_processed}, Annotated saved: {num_annotated_saved}, Errors: {num_errors}")



[START] Annotating videos
- Source: MP_Videos_All
- Annotated out: MP_Videos_All_annotated


I0000 00:00:1761471775.366544 57825104 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
W0000 00:00:1761471775.436166 59532079 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761471775.444898 59532079 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761471775.446642 59532079 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761471775.446658 59532077 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761471775.446956 59532080 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disablin

[DONE] Videos processed: 180, Annotated saved: 180, Errors: 0


## 5. Collect Keypoint Values for Training and Testing

In [56]:

# Source (augmented) videos
source_dir = 'MP_Videos_All'
if not os.path.isdir(source_dir):
    raise FileNotFoundError(f"Source directory not found: {source_dir}")

# Output structure: per-place / per-video / per-frame.npy
keyframes_root = f"{source_dir}_keyframes"
os.makedirs(keyframes_root, exist_ok=True)

supported_exts = {'.mp4', '.avi', '.mov', '.mkv'}


def extract_keypoints(results):
    pose = np.zeros(33 * 4)
    left_hand = np.zeros(21 * 3)
    right_hand = np.zeros(21 * 3)
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark]).flatten()
    if results.left_hand_landmarks:
        left_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten()
    if results.right_hand_landmarks:
        right_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten()
    return np.concatenate([pose, left_hand, right_hand])


num_videos = 0
num_frames_saved = 0
num_errors = 0

print(f"[START] Extracting per-frame keypoints\n- Source: {source_dir}\n- Output root: {keyframes_root}")

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for place in sorted(os.listdir(source_dir)):
        place_src = os.path.join(source_dir, place)
        if not os.path.isdir(place_src):
            continue
        place_out = os.path.join(keyframes_root, place)
        os.makedirs(place_out, exist_ok=True)

        for fname in sorted(os.listdir(place_src)):
            ext = os.path.splitext(fname)[1].lower()
            if ext not in supported_exts:
                continue

            src_path = os.path.join(place_src, fname)

            cap = cv2.VideoCapture(src_path)
            if not cap.isOpened():
                print(f"[WARN] Could not open video: {src_path}")
                num_errors += 1
                continue

            # Create per-video folder under place
            video_name = os.path.splitext(fname)[0]
            vid_out_dir = os.path.join(place_out, video_name)
            os.makedirs(vid_out_dir, exist_ok=True)

            frame_idx = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break

                _, results = mediapipe_detection(frame, holistic)
                kps = extract_keypoints(results)
                np.save(os.path.join(vid_out_dir, f"{frame_idx}.npy"), kps)
                frame_idx += 1
                num_frames_saved += 1

            cap.release()
            num_videos += 1

print(f"[DONE] Videos processed: {num_videos}, Frames saved: {num_frames_saved}, Errors: {num_errors}")



[START] Extracting per-frame keypoints
- Source: MP_Videos_All
- Output root: MP_Videos_All_keyframes


I0000 00:00:1761472239.471400 57825104 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Max
W0000 00:00:1761472239.538714 59571689 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761472239.549972 59571689 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761472239.552234 59571691 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761472239.552297 59571697 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1761472239.552500 59571689 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disablin

[DONE] Videos processed: 180, Frames saved: 10800, Errors: 0


## 5. Check Shape

In [58]:
import os
import numpy as np
import cv2

kp_root = 'MP_Videos_All_keyframes'
ann_root = 'MP_Videos_All_annotated'

# Find one .npy
sample_kp = None
for root, _, files in os.walk(kp_root):
    for f in files:
        if f.lower().endswith('.npy'):
            sample_kp = os.path.join(root, f)
            break
    if sample_kp:
        break

if sample_kp and os.path.isfile(sample_kp):
    try:
        arr = np.load(sample_kp)
        print('[KEYPOINTS] File:', sample_kp)
        print(' - shape:', arr.shape)
        print(' - dtype:', arr.dtype)
    except Exception as e:
        print(f'[KEYPOINTS] Error loading {sample_kp}:', str(e))
else:
    print('[KEYPOINTS] No .npy file found under', kp_root)

# Find one annotated video
video_exts = {'.mp4', '.avi', '.mov', '.mkv'}
sample_vid = None
for root, _, files in os.walk(ann_root):
    for f in files:
        if os.path.splitext(f)[1].lower() in video_exts:
            sample_vid = os.path.join(root, f)
            break
    if sample_vid:
        break

if sample_vid and os.path.isfile(sample_vid):
    cap = cv2.VideoCapture(sample_vid)
    try:
        if cap.isOpened():
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            print('\n[ANNOTATED VIDEO] File:', sample_vid)
            print(' - resolution:', f'{width}x{height}')
            print(' - fps:', fps)
            print(' - frame_count:', count)
        else:
            print('[ANNOTATED VIDEO] Could not open:', sample_vid)
    except Exception as e:
        print(f'[ANNOTATED VIDEO] Error reading {sample_vid}:', str(e))
    finally:
        cap.release()
else:
    print('[ANNOTATED VIDEO] No video found under', ann_root)


[KEYPOINTS] File: MP_Videos_All_keyframes/Bedok/20/20.npy
 - shape: (258,)
 - dtype: float64

[ANNOTATED VIDEO] File: MP_Videos_All_annotated/Bedok/7.mp4
 - resolution: 1920x1080
 - fps: 30.0
 - frame_count: 60
